# Retrieve USGS WDFN data

*Federal Open Water Data for Researchers* (CUAHSI), session 5.

**Learning goal:** Retrieve USGS WDFN data for a location of interest, describe what comes back using the course's shared concepts, and choose an access pattern that scales with your question.

**Environment:** run this notebook in the `m03-wdfn` conda environment, created from `module-03_lesson-03_retrieve-usgs-wdfn-data_environment.yml` in this folder (see the bundle README). Run the cells in order: later cells reuse variables from earlier ones.

**Student notebook.** "Now you try it" cells are starters: fill in the `...` blanks, guided by the `# TODO:` comments. The solutions notebook in the same folder has an example answer.

USGS Water Data for the Nation (WDFN) publishes the observations USGS collects at thousands of monitoring locations across the country: river discharge and gage height, groundwater levels, water quality and more. This lesson shows you how to find monitoring locations and retrieve their streamflow data in Python. For an introduction to the data, how they're collected, and their limitations, start with the Module 2 reading *Meet USGS WDFN*.

You will find the stream monitoring locations around the Ohio River at Louisville, Kentucky, then retrieve the sensor record, daily values and field measurements from the gage on the Ohio, monitoring location `USGS-03294500`, for 15 March to 15 May 2025.

**A note on names.** The data you'll retrieve are stored in the National Water Information System (NWIS), the USGS database behind WDFN. Many older tutorials say "NWIS" for the whole system, and its older web services carry that name. USGS currently runs two generations of web service side by side:

## Attribution and licence

This notebook is part of *Federal Open Water Data for Researchers*, a CUAHSI course, and is released under the [GNU General Public License v3.0](https://www.gnu.org/licenses/gpl-3.0.html) (GPL-3.0). You may use, change and share it under the same licence.

It adapts code and ideas from these sources:

- Adapted from [Notebook to Demonstrate Collecting USGS Data](https://github.com/CUAHSI/notebooks/tree/develop/Data%20Access%20Examples/USGS%20-%20Plotting%20Streamflow%20using%20NWIS%20DataRetrieval) by CUAHSI, CUAHSI notebooks (GPL-3.0). Its legacy `nwis` calls are ported to `waterdata` here.

Other sources used in the lesson are listed under "Further reading" at the end of this notebook.

## Credentials

Credentials come only from your account or environment, never from the notebook itself.

**USGS Water Data API key (optional).** The USGS Water Data APIs work without a key, but a key raises your rate limit. Request one at the [USGS Water Data API signup page](https://api.waterdata.usgs.gov/signup/). `dataretrieval` reads it from the `API_USGS_PAT` environment variable and sends it for you. If your hosted environment doesn't set that variable, uncomment the two lines below and paste your key when asked: it is kept only for this session and isn't saved in the notebook. Don't print the key or write it into a cell.

In [ ]:
import os

# Optional: set the key for this session only (it isn't saved in the notebook)
# from getpass import getpass
# os.environ["API_USGS_PAT"] = getpass("USGS API key: ")
print("API_USGS_PAT is set:", bool(os.getenv("API_USGS_PAT")))

## Choosing an access route

All the routes below use the same Water Data APIs, so the choice is mostly about *which function* matches your question. None of them needs a key, but USGS recommends one for repeated or large requests. If you do not have an API key your queries have a limit of 50 requests per IP address per hour. With an API key that limit increases to 1000 requests per IP address per hour.

| Use case | Recommended route | Key needed? | Scaling limits |
|---|---|---|---|
| Explore by clicking, spot-check a site | [WDFN website](https://waterdata.usgs.gov/) and [Water Data For The Nation - Explore](https://waterdata.usgs.gov/explore/) | No | Manual; not reproducible on its own |
| Find monitoring locations and what they measure | `waterdata.get_monitoring_locations`, `get_time_series_metadata`, `get_combined_metadata` | No (key recommended) | Results come in pages with a maximum of 50,000 rows per page; each page is one request |
| Daily values (e.g. daily mean discharge), any length of record | `waterdata.get_daily` | No (key recommended) | The whole record for a site in one query (paged for you) |
| The sensor record (every 5–15 minutes) | `waterdata.get_continuous` | No (key recommended) | Up to three years per query |
| Discharge measured by hydrographers in the field | `waterdata.get_field_measurements` | No (key recommended) | About one measurement a month per site |
| One day (or a short window) at many sites | `waterdata.get_daily` with no sites, a list of sites, or optionally a `bbox` | No (key recommended) | Fewer requests than iterating by site; mind the page/request count |
| Code that uses the legacy Water Services | `dataretrieval.nwis` | No | Being retired; port it to `waterdata` (see the note below) |

## Tools and environment setup

This notebook runs in the `m03-wdfn` conda environment, made from `module-03_lesson-03_retrieve-usgs-wdfn-data_environment.yml` in this folder (the bundle README says how). Credentials are set up in the Credentials section above.

## Programmatic data discovery

### Codes: parameters and statistics

The Water Data APIs use codes for many query arguments. A parameter code says what was measured (`00060` is discharge, `00065` is gage height), and a statistic code says how values were summarized (`00003` is the daily mean). `get_reference_table` returns each list of codes as a `pandas` DataFrame, so you can look them up in code rather than on a web page. Like every `waterdata` function, it returns a pair, `(DataFrame, metadata)`; `_` or `md` catches the metadata:

In [ ]:
from dataretrieval import waterdata

parameter_codes, _ = waterdata.get_reference_table("parameter-codes")
statistic_codes, _ = waterdata.get_reference_table("statistic-codes")

print(parameter_codes.loc[parameter_codes["parameter_code"].isin(["00060", "00065"]),
                          ["parameter_code", "parameter_name", "unit_of_measure"]])
print(statistic_codes.loc[statistic_codes["statistic_code"].isin(["00003", "00011"])])

`parameter_codes` has about 19,600 rows and `statistic_codes` about 3,200. Other reference tables work the same way, for example `"site-types"`, `"states"`, `"counties"`, `"hydrologic-unit-codes"` and `"agency-codes"`.

**Note**: the `unit_of_measure` for `Discharge` above says `ft3/s`, but when retrieving data, the unit is referred to as `ft^3/s`.

### Finding monitoring locations

**The point-and-click way.** The [WDFN website](https://waterdata.usgs.gov/) and [Water Data For The Nation - Explore](https://waterdata.usgs.gov/explore/) let you search by place name, address, state or watershed and click on sites. That's fine for exploring. A programmatic discovery step keeps your search reproducible, so capture it in code too.

Keep what you're asking for (place, site type, codes, dates) in variables at the top of your script, rather than typing them into the middle of each function call. Your query is then recorded in one spot and easy to change.

**Example: which stream monitoring locations are in Jefferson County, Kentucky (Louisville)?**

In [ ]:
# what we're asking for
STATE, COUNTY = "Kentucky", "Jefferson County"
site = "USGS-03294500"                    # Ohio River at Louisville, KY (found below)
START, END = "2025-03-15", "2025-05-15"   # the April 2025 flood

site_info, md = waterdata.get_monitoring_locations(
    state_name=STATE,
    county_name=COUNTY,
    site_type="Stream",
)
print(site_info.shape)
print(site_info[["monitoring_location_id", "monitoring_location_name", "drainage_area"]].head())

This returns a `GeoDataFrame` with one row per monitoring location (120 on 2026-10-08) and 44 columns. Key columns are `monitoring_location_id` (the *Location identifier*), `monitoring_location_name`, `site_type`, `drainage_area` (square miles) and `geometry` (a point you can map). The list includes inactive sites, and, as the first rows show, a few entries that aren't usable gages: on 2026-10-08, ten "STUDENT" creeks and five "TEST" entries (such as "SANDY TEST NO 1" and "LOUISVILLE TEST SITE"). This is one reason to look at discovery results before you download anything.

**Example: which of those have daily mean discharge?**

`get_combined_metadata` can be used as an alternative to `get_monitoring_locations` (used above) to discover monitoring locations. It combines monitoring-location details with the list of time series each location records, so you can filter on parameter and statistic codes in the same query.

In [ ]:
sites_with_q, md = waterdata.get_combined_metadata(
    state_name=STATE,
    county_name=COUNTY,
    site_type="Stream",
    parameter_code="00060",   # discharge
    statistic_id="00003",     # daily mean
)
print(len(sites_with_q), "time series at", sites_with_q["monitoring_location_id"].nunique(), "locations")
print(sites_with_q[["monitoring_location_id", "monitoring_location_name", "begin", "end"]].head())

Each row is one *time series*, not one location, so a location with two daily-discharge series appears twice (here, 29 rows for 27 locations; `USGS-03292555` in row 1 is one of them). `begin` and `end` show the period each series covers: a series whose `end` is years ago belongs to a gage that no longer reports. One of these is `USGS-03294500`, Ohio River at Louisville, KY, with daily discharge from 1928 to the present.

**Map: where these locations are**

A quick map is a good check that discovery found what you meant. The river line on this map comes from the USGS NLDI web service, which returns the stream network up- and downstream of a monitoring location as GeoJSON:

In [ ]:
import pandas as pd
import geopandas as gpd
import requests
import matplotlib.pyplot as plt

# site (Ohio River at Louisville) is set in the query block above

# leave out the 15 entries that are not real gages (names such as "STUDENT1 CREEK" or "SANDY TEST NO 1")
real = site_info[~site_info["monitoring_location_name"].str.contains("STUDENT|TEST")]
with_q = sites_with_q.drop_duplicates("monitoring_location_id")

# the Ohio River, 60 km up- and downstream of the gage (UM = upstream mainstem, DM = downstream mainstem)
parts = []
for nav in ["UM", "DM"]:
    url = f"https://api.water.usgs.gov/nldi/linked-data/nwissite/{site}/navigation/{nav}/flowlines"
    r = requests.get(url, params={"distance": 60}, timeout=120)
    r.raise_for_status()
    parts.append(gpd.GeoDataFrame.from_features(r.json()["features"], crs="EPSG:4326"))

river = pd.concat(parts)
fig, ax = plt.subplots(figsize=(7, 6))
river.plot(ax=ax, color="#2a78d6", linewidth=2)
real.plot(ax=ax, color="#b9b8b2", markersize=14, label=f"Stream monitoring locations ({len(real)})")
with_q.plot(ax=ax, color="#52514e", markersize=24, label=f"With daily mean discharge ({len(with_q)})")
gage = with_q[with_q["monitoring_location_id"] == site]
gage.plot(ax=ax, color="#eb6834", markersize=220, marker="*", edgecolor="white", linewidth=0.8, zorder=5,
          label=f"{site}, Ohio River at Louisville, KY")
ax.annotate("Ohio River", (-85.97, 38.03), fontsize=10, color="#2a78d6", style="italic")
ax.annotate("Louisville gage", gage.geometry.iloc[0].coords[0], xytext=(-100, 14), textcoords="offset points", fontsize=10)
ax.set_xlim(-86.05, -85.35)
ax.set_ylim(37.95, 38.42)
ax.set_aspect(1 / 0.786)   # roughly equal km in x and y at 38° N
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("USGS stream monitoring locations, Jefferson County, KY", fontsize=12, loc="left")
ax.legend(loc="lower right", fontsize=9)
fig.tight_layout()
fig.savefig("wdfn-ohio-louisville-map.png", dpi=150)

**Example: what does the Louisville gage record?**

Once you've picked a location, ask which time series it records. This is discovery for a single site:

In [ ]:
series, md = waterdata.get_time_series_metadata(monitoring_location_id=site)
print(series[["parameter_code", "parameter_name", "statistic_id", "computation_period_identifier", "begin", "end"]])

Each row is one time series. `statistic_id` `00011` with period `Points` is the continuous sensor record (gage height since 2007, discharge since 2009), and `00003` with period `Daily` is the daily mean (discharge since 1928). The Water Year rows are annual summaries (they start in 1832, long before the sensor record), the sediment series ran only from 1979 to 1983, and `00045` is a precipitation record added in June 2026. Your `end` dates will be later.

Once you have your monitoring location ID(s) and codes, move on to downloads.

## Programmatic data downloads

Every `waterdata` function returns a `(DataFrame, metadata)` pair. The DataFrame holds the values; the metadata object records the request (`md.url`) and when it was made, which is useful when you write down how you got your data.

### Continuous data

**Continuous values** are the sensor record, typically every 15 minutes. `get_continuous` accepts up to three years per query. Here we request two months of discharge and gage height together:

In [ ]:
# START and END (the April 2025 flood) are set in the query block above
cont, md = waterdata.get_continuous(
    monitoring_location_id=site,
    parameter_code=["00060", "00065"],    # discharge and gage height
    time=f"{START}T00:00:00Z/{END}T23:59:59Z",
)
print(cont.shape)
print(cont[["time", "parameter_code", "value", "unit_of_measure", "approval_status", "qualifier"]].head())

The result is a long-format `GeoDataFrame`: one row per timestamp and parameter (about 5,850 rows for each of the two parameters, every 15 minutes), with columns including `monitoring_location_id`, `parameter_code`, `statistic_id`, `time` (UTC), `value`, `unit_of_measure`, `approval_status` and qualifier.

To find the flood crest, take the row with the largest value for each parameter:

In [ ]:
peaks = cont.loc[cont.groupby("parameter_code")["value"].idxmax()]
print(peaks[["parameter_code", "time", "value", "unit_of_measure", "approval_status"]])

The Ohio crested at **719,000 ft³/s** at 14:30 UTC on 9 April 2025 (10:30 a.m. in Louisville), with a gage height of **67.98 ft** half an hour earlier. Gage height is measured above the gage's own reference point (its datum), not from the riverbed, so it isn't the water depth. Times are in UTC: convert them before comparing with local records.

Look at `approval_status`, too. On 2026-10-08, values up to 9 April 2025 17:15 UTC were `Approved`, and everything after that was still provisional. Provisional data can still be revised when USGS reviews them ([USGS provisional data statement](https://waterdata.usgs.gov/provisional-data-statement/)), so check `approval_status` before you publish numbers, and expect values after the crest to change. No values in this window carry a `qualifier`.

### Daily data

**Daily values** are summaries of the continuous record, here the daily mean (`statistic_id="00003"`) discharge. The `time` argument can be a plain date range:

In [ ]:
daily, md = waterdata.get_daily(
    monitoring_location_id=site,
    parameter_code="00060",
    statistic_id="00003",     # daily mean
    time=f"{START}/{END}",
)
daily = daily.sort_values("time")
print(daily["approval_status"].value_counts().to_dict())
print(daily[["time", "value", "unit_of_measure", "approval_status", "qualifier"]].iloc[[0, 17, 25, 30, 61]])

There are 62 daily values, one per day. The daily mean peaked at 712,000 ft³/s on 9 April, slightly below the 719,000 ft³/s continuous peak, as expected for an average. That day is `Provisional` even though its morning values are approved, because the daily mean includes the provisional afternoon. Daily `time` is the monitoring location's local calendar date, not a timestamp.

### Field  measurements

**Field measurements** are the discharge, gage-height, and groundwater level measurements that hydrographers make in person at the gage. USGS uses discharge and gage-height to build and check the rating curve that turns the sensor's gage height into the continuous discharge record, so they are the closest thing to "ground truth" for discharge:

In [ ]:
fm, md = waterdata.get_field_measurements(
    monitoring_location_id=site,
    time="2025-01-01T00:00:00Z/2025-12-31T00:00:00Z",
)
discharge_fm = fm[fm["parameter_code"] == "00060"].sort_values("time")
print(discharge_fm[["time", "value", "unit_of_measure", "observing_procedure", "measurement_rated", "approval_status"]])

Field measurements include both discharge (`00060`) and gage-height (`00065`) readings; we kept only discharge. In 2025, hydrographers measured discharge five times, including on the day of the crest. `observing_procedure` records how each measurement was made, here always with an acoustic Doppler current profiler (ADCP) towed across the river. `measurement_rated` is a qualitative estimate of the quality of a measurement: the hydrographer's judgment of its accuracy. The crest-day measurement (738,000 ft³/s) is rated `Poor`, which is common for measurements in fast, debris-laden flood water. It is about 3% above the continuous record's peak, well within what a `Poor` rating allows.

### Plot the data

**What the data look like.** Plotting the continuous record by approval status, with the field measurement on top, shows the flood, the crest and where provisional data begin:

In [ ]:
import matplotlib.dates as mdates

fig, ax = plt.subplots(figsize=(8, 4.2))
q = cont[cont["parameter_code"] == "00060"].sort_values("time")
colors = {"Approved": "#2a78d6", "Provisional": "#eb6834"}
for status, part in q.groupby("approval_status"):
    ax.plot(part["time"], part["value"] / 1000, color=colors[status], linewidth=2, label=f"Continuous discharge ({status})")

in_window = discharge_fm[(discharge_fm["time"] >= START) & (discharge_fm["time"] <= END)]
ax.scatter(in_window["time"], in_window["value"] / 1000, s=60, color="#0b0b0b", zorder=5, label="Field measurement")
crest = q.loc[q["value"].idxmax()]
ax.annotate(f"Crest {crest['value']:,.0f} ft³/s\n{crest['time']:%d %b %H:%M} UTC", (crest["time"], crest["value"] / 1000),
            xytext=(18, -8), textcoords="offset points", fontsize=9)
ax.set_ylabel("Discharge (thousand ft³/s)")
ax.set_ylim(0, 800)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax.set_title(f"Ohio River at Louisville, KY ({site}), spring 2025", fontsize=12, loc="left")
ax.legend(frameon=False, fontsize=9, loc="center right", bbox_to_anchor=(1.0, 0.62))
fig.tight_layout()
fig.savefig("wdfn-ohio-louisville-discharge.png", dpi=150)

**A note on the legacy `nwis` module.** Many older tutorials, including the CUAHSI notebook this lesson draws on, use `dataretrieval.nwis`. This calls the legacy Water Services which will be [decommissioned February 22nd, 2027](https://waterdata.usgs.gov/blog/wdfn-waterservices-degradation/).

For those curious, here are some differences in the data returned by legacy `nwis` when compared to the modernized `waterdata` module. The values are the same, but the shape differs: a bare site number (`03294500` instead of `USGS-03294500`), one column per parameter and statistic (`00060_Mean`), and approval as a one-letter code (`A` approved, `P` provisional) in a `_cd` column.

## Understanding what you downloaded

The continuous values, daily values and field measurements share their core columns. Here is how they map onto the course's shared concepts (see the Module 2 reading *Meet USGS WDFN*), using what came back for Louisville:

| Shared concept | What it is in your download | Column(s) |
|---|---|---|
| Location identifier | `USGS-03294500`: agency prefix plus site number | `monitoring_location_id` |
| Variable | discharge (`00060`) and gage height (`00065`) | `parameter_code`, plus `statistic_id` (`00011` continuous, `00003` daily mean) |
| Variable unit | `ft^3/s` for discharge, `ft` for gage height | `unit_of_measure` |
| Time | continuous: UTC timestamps every 15 minutes; daily: a calendar date; field measurements: a date plus `time_of_day` | `time` |
| Data quality flag(s) | approval status (`Approved`, or `Provisional` and subject to revision); `qualifier` for special conditions (e.g. `[ESTIMATED]`, `[ICE]`); `measurement_rated` (`Good`, `Fair`, `Poor`) for field measurements | `approval_status`, `qualifier`, `measurement_rated` |
| Version / provenance | No version number: the service you called, the request URL and the date you retrieved the data. Values can change until they're approved; `last_modified` says when each was last changed | `md.url`, your retrieval date, `last_modified` |
| Data unit | one time series per monitoring location, parameter and statistic | `time_series_id` |

**Provisional data and re-running later.** Because provisional values can change, the same request run next year may return different numbers for April 2025. Record the date you retrieved the data, and save what you downloaded.

**Keep raw and derived data apart.** Save each download unchanged (for example `data/raw/usgs_03294500_continuous_2025-03-15_2025-05-15.csv`), and write anything you compute from it, such as the crest table or a cleaned series, to a separate folder (`data/derived/`). Then you can always trace a number back to the download it came from (Data Management, Module 1).

**Cite what you used.** USGS water data have a DOI, [https://doi.org/10.5066/F7P55KJN](https://doi.org/10.5066/F7P55KJN) (U.S. Geological Survey, National Water Information System). Cite it with the monitoring location IDs, parameter codes, dates and your access date, and cite `dataretrieval` too: Hodson and Hariharan, [https://doi.org/10.5066/P94I5TX3](https://doi.org/10.5066/P94I5TX3). Data Publishing in Module 1 covers citing data.

## Best practices FAQs

See the sections below for answers and code examples to the following questions.

* What is the recommended way to download data for **one location across the full period of record**?
* What is the recommended way to download data across **all locations for a small time range**?
* If I am working on improving efficiency through **code parallelization**, what should I do vs avoid?

### Temporal scaling

**What is the recommended way to access data for one location but the full period of record?**

Make one request per site and leave out `time`. For daily values, `get_daily` then returns the whole record, and `dataretrieval` handles the paging. The Louisville gage has daily discharge back to 1928:

In [ ]:
daily_record, md = waterdata.get_daily(
    monitoring_location_id=site,
    parameter_code="00060",
    statistic_id="00003",
)
print(len(daily_record), daily_record["time"].min(), daily_record["time"].max())
print(daily_record.loc[daily_record["value"].idxmax(), ["time", "value"]])

On 2026-10-08 this one request returned 36,054 rows, one per day from 1 January 1928 to the day before, in about 6 seconds. The largest daily mean in the record is 1,110,000 ft³/s on 27 January 1937. Look at the `qualifier` column on long records: about 1,100 of these days are marked `[ESTIMATED]` (count them with `daily_record["qualifier"].astype(str).str.contains("ESTIMATED").sum()`), some together with `[ICE]` or `[EQUIP]` (equipment problems).

Continuous values are limited to three years per query, so request a long continuous record in three-year windows (the Louisville record starts in 2009, so that's six chunks of calls). Approved data rarely change, so save each window to a file and only request new data next time.

### Spatial scaling

**What is the recommended way to download data across all locations but a small time range?**

Leave out `monitoring_location_id` and set a short `time` instead. One request for one day returns that day's value for every site with daily mean discharge. Looping over thousands of site IDs one at a time would quickly use up your rate limit.

In [ ]:
one_day, md = waterdata.get_daily(
    parameter_code="00060",
    statistic_id="00003",
    time="2025-04-09",    # one date; also accepts an interval such as "2025-04-01/2025-04-10"
)
print(len(one_day), "rows at", one_day["monitoring_location_id"].nunique(), "locations")

On 2026-10-08 this returned 9,041 daily values at 9,029 locations (a few locations have more than one discharge series) in a couple of seconds. To narrow the area, add a `bbox` (west, south, east, north, in decimal degrees) or a list of sites rather than looping:

In [ ]:
around_louisville, md = waterdata.get_daily(
    parameter_code="00060",
    statistic_id="00003",
    time="2025-04-09",
    bbox=[-86.5, 37.8, -85.0, 38.8],
)
print(around_louisville.sort_values("value", ascending=False)[["monitoring_location_id", "value", "approval_status"]].head(3))

That returned 36 locations in the box. The two largest are the Louisville gage and a second Ohio River location in Louisville, at the water tower.

### Parallelization

If I am working on improving efficiency of my code through parallelization, what should I do vs avoid?

- **Do** ask for more in each request before reaching for parallel code. Most `waterdata` functions accept lists (several `monitoring_location_id`s or `parameter_code`s), a `bbox`, or a time interval. One request that returns 10,000 rows is cheaper for you and for USGS than 100 requests that return 100 rows each.
- **Do** use an API key (`API_USGS_PAT`) for any repeated or large workflow, and expect HTTP `429 Too Many Requests` if you go over your hourly limit.
- **Do** let `dataretrieval` handle paging (`limit` sets the page size, `max_rows` caps the total) instead of writing your own loop of requests.
- **Avoid** launching many simultaneous requests from your own threads or processes. Each request spends your rate-limit quota, and a burst of parallel calls is the quickest way to get throttled. `dataretrieval` will automatically chunk queries that go over the byte limit and run them in parallel. Parallelization settings can be adjusted, but is not recommended.
- **Avoid** re-downloading the same historical record every time you run your code. Approved data rarely change, so save results to a file and only request what is new (the `last_modified` argument helps).

## Now you try it

Repeat the main steps for accessing USGS WDFN data on a second river: the **Willamette River at Salem, Oregon**.

**Your task:** Retrieve discharge from a monitoring location on the Willamette using `dataretrieval`.

Compile relevant code from above to create a small, stand-alone code chunk that performs the steps below. Give the new results new names (for example, adding `_salem`), so you can still compare them with the Ohio River at Louisville.

1. Change `site` to `USGS-14191000`.
2. Get the time series metadata for the monitoring location to check the expected record lengths.
3. Change the dates `START, END` to `"2026-02-01", "2026-03-31"`
4. Get the daily mean discharge for 1 February to 31 March 2026. Find the highest daily mean, and check its approval status.
5. Get the continuous discharge and gage height for the same window and find the crest, as in the continuous-values example.

**What you should see:** You should see discharge records going back to 1989 for continuous values, and 1909 for daily values. From 1 February to 31 March there are 59 daily values, almost all `Provisional`; the highest daily mean is **62,200 ft³/s** on 25 February 2026. The continuous record crests at **64,000 ft³/s** (gage height 16.86 ft) at 23:45 UTC on 25 February. Because these values are provisional, your numbers may differ slightly.

**In this notebook:** make the changes in the code cell below rather than editing the earlier cells. It sets the shared variables (such as the place and dates) to the new river's values, so to go back to the earlier examples, run the notebook again from the top.

In [ ]:
# This cell raises an error until every ... is filled in.
site = ...          # TODO: the Willamette River at Salem monitoring location ID from the task
START, END = ...    # TODO: the new dates from the task

series_w, md = waterdata.get_time_series_metadata(monitoring_location_id=site)
print(series_w[["parameter_code", "parameter_name", "statistic_id", "computation_period_identifier", "begin", "end"]])
daily_w, md = waterdata.get_daily(monitoring_location_id=site, parameter_code=...,
                                  statistic_id=..., time=f"{START}/{END}")   # TODO: daily mean discharge
print(len(daily_w), daily_w["approval_status"].value_counts().to_dict())
print(daily_w.loc[..., ["time", "value", "unit_of_measure", "approval_status"]])   # TODO: the row with the highest value

cont_w, md = waterdata.get_continuous(monitoring_location_id=site, parameter_code=...,   # TODO: discharge and gage height
                                      time=f"{START}T00:00:00Z/{END}T23:59:59Z")
peaks_w = ...   # TODO: the row with the largest value for each parameter_code
print(peaks_w[["parameter_code", "time", "value", "unit_of_measure", "approval_status"]])

## Going further

*Optional.* These examples go beyond the session. Nothing earlier in the notebook depends on them, but they reuse variables and functions from the cells above, so run those first.

### Counting qualifiers on long records

Long records carry more qualifiers. In the "Temporal scaling" example, about 1,100 days of the whole daily record (`daily_record`) are marked `[ESTIMATED]`, some together with `[ICE]` or `[EQUIP]` (equipment problems). Run that example first, then count them:

In [ ]:
print(daily_record["qualifier"].astype(str).str.contains("ESTIMATED").sum(), "days marked [ESTIMATED]")
daily_record["qualifier"].astype(str).value_counts()

## Further reading

* Meet USGS WDFN (Module 2): what these data are and how they're collected.
* `dataretrieval` (Python) GitHub repo: https://github.com/DOI-USGS/dataretrieval-python
* `dataretrieval` documentation: https://doi-usgs.github.io/dataretrieval-python/
* USGS Water Data APIs documentation: https://api.waterdata.usgs.gov/
* USGS Water Data API keys: https://api.waterdata.usgs.gov/docs/ogcapi/keys/
* USGS Water Data for the Nation: https://waterdata.usgs.gov/
* Water Data For The Nation - Explore (GUI): https://waterdata.usgs.gov/explore/
* USGS provisional data statement: https://waterdata.usgs.gov/provisional-data-statement/
* NWIS WaterServices decommissioning: https://waterdata.usgs.gov/blog/wdfn-waterservices-degradation/
* Adapted from [Notebook to Demonstrate Collecting USGS Data](https://github.com/CUAHSI/notebooks/tree/develop/Data%20Access%20Examples/USGS%20-%20Plotting%20Streamflow%20using%20NWIS%20DataRetrieval) by CUAHSI, CUAHSI notebooks (GPL-3.0). Its legacy `nwis` calls are ported to `waterdata` here.